In [1]:
# Imports
import time

# Dependencies
import pandas as pd

# Library
from api_builder.fsm import menus
from api_builder.fsm.states import State
from api_builder.fsm.menus import NATIONS
from api_builder.fsm import fsm
from utils.utils import construct_rules
from api_builder.fsm.fsm_utils import parse_links
from api_builder.fsm.fetch import fetch_routine
from pipeline.builder import VanguardPipeline
from pipeline.query_builder import make_query
from data.vanguard_data import VanguardStorage
from api_builder.fsm.scrap import main_scrap_routine
from parsers.vanguard_parser import VanguardParser
from classifier.vanguard_classifier import VanguardClassifier
from api_builder.vanguard_api_build import MediaWikiAPI, VanguardScrapper
from api_builder.api_request import header


In [2]:
# Url Parser
web = MediaWikiAPI()
pipeline = VanguardPipeline(
	VanguardScrapper(web),
	VanguardParser(),
	VanguardClassifier(),
	VanguardStorage()
)
await pipeline.scrapper.api.init_session()
state_machine = fsm.FSMContext()
state = State.ENTRY_POINT
while (state != State.END):
	if (state == State.ENTRY_POINT):
		state = menus.entry_point(state_machine)
	elif (state == State.SELECT_MAIN_CATEGORY):
		state = menus.select_category(state_machine)
	elif (state == State.SELECT_SUBCATEGORY):
		state = menus.select_subcategory(state_machine)
	elif (state == State.BUILD_QUERY):
		state = make_query(state_machine)
	elif (state == State.FETCH):
		state = await fetch_routine(state_machine, pipeline.scrapper)
	elif (state == State.PARSE):
		pipeline.classifier._define_rules(construct_rules(state_machine.main_category.capitalize()))
		state = parse_links(
			state_machine, pipeline.parser, pipeline.storage,
			pipeline.scrapper, pipeline.classifier
		)
time.sleep(4)

Welcome to VanguardTCGScrapper


What info do you need from the website?
0 : boosters
1 : specials
2 : decks
3 : others
You selected: boosters
Which subcategory you want to scrap?
indentify it with the index number:

0:  Booster Sets
1:  Extra Booster Sets
2:  Character Booster Sets
3:  Clan Booster Sets
4:  Title Booster Sets
5:  Unique Booster Sets


In [ ]:
from fsm.routines.check_data_base import build_set_path

for block in ["LB", "LL", "G", "V", "D", "DZ"]:
	consult = pipeline.parser.make_consults(getattr(pipeline.storage, block.lower()))
	print(consult)
	for tpl in consult.values():
		print(f"voy a pasar esta consulta: {tpl}")
		time.sleep(6)
		api_result = await pipeline.scrapper.api.get(
			params=tpl,
			headers=header
		)
		print(api_result)
		wikitex = pipeline.scrapper.obtain_wikitex(api_result)
		data = pipeline.scrapper.make_cardlist_from_str(wikitex=wikitex)
		infobox = pipeline.parser.infobox(wikitex)
		is_d = False
		is_deck = False
		if (block in ["D", "DZ"]):
			is_d = True
		rows = pipeline.storage.prepare_data(data, 6, is_d=is_d, is_deck=is_deck, infobox=infobox)
		df = pd.DataFrame(rows, columns=[
			"CardNo", "Name", "Grade", "Faction",
			"FactionType", "Type", "Rarity", "Release"
		])
		set_number = pipeline.classifier.obtain_set_number(
			data[0]
		)
		path = build_set_path(
			category="boosters",
			set_type="main",
			block=block,
			set_number=set_number
		)
		path.parent.mkdir(
			parents=True,
			exist_ok=True
		)
		df.to_parquet(path)

		print(data)

In [ ]:
db = pd.read_parquet("data/database/boosters/booster sets/d/set_08.parquet")
db.loc[db["CardNo"] == "None"]

In [ ]:
db[90:100]

In [ ]:
db[119:210]

In [ ]:
pipeline.storage.lb

In [3]:
dz_consults = pipeline.parser.make_consults(pipeline.storage.v, "consult")

In [4]:
dz_consults

{0: {'action': 'query',
  'format': 'json',
  'prop': 'revisions',
  'titles': 'V Booster Set 01: Unite! Team Q4',
  'rvprop': 'content',
  'rvslots': 'main'},
 1: {'action': 'query',
  'format': 'json',
  'prop': 'revisions',
  'titles': 'V Booster Set 02: Strongest! Team AL4',
  'rvprop': 'content',
  'rvslots': 'main'},
 2: {'action': 'query',
  'format': 'json',
  'prop': 'revisions',
  'titles': 'V Booster Set 03: Miyaji Academy Cardfight Club',
  'rvprop': 'content',
  'rvslots': 'main'},
 3: {'action': 'query',
  'format': 'json',
  'prop': 'revisions',
  'titles': 'V Booster Set 04: Vilest! Deletor',
  'rvprop': 'content',
  'rvslots': 'main'},
 4: {'action': 'query',
  'format': 'json',
  'prop': 'revisions',
  'titles': 'V Booster Set 05: Aerial Steed Liberation',
  'rvprop': 'content',
  'rvslots': 'main'},
 5: {'action': 'query',
  'format': 'json',
  'prop': 'revisions',
  'titles': 'V Booster Set 06: Phantasmal Steed Restoration',
  'rvprop': 'content',
  'rvslots': 'main

In [5]:
dz_consults[0]

{'action': 'query',
 'format': 'json',
 'prop': 'revisions',
 'titles': 'V Booster Set 01: Unite! Team Q4',
 'rvprop': 'content',
 'rvslots': 'main'}

In [6]:
api_answer = await pipeline.scrapper.api.get(dz_consults[0], headers=header)

In [ ]:
api_answer["parse"]["text"]["*"]

In [25]:
api_answer

{'parse': {'title': 'V Booster Set 01: Unite! Team Q4',
  'pageid': 1765690,
  'links': [{'ns': 0,
    'exists': '',
    '*': 'G Booster Set 14: Divine Dragon Apocrypha'},
   {'ns': 0, 'exists': '', '*': 'Set Gallery:VG-V-BT01'},
   {'ns': 0, 'exists': '', '*': 'Set Gallery:VGE-V-BT01'},
   {'ns': 0, 'exists': '', '*': 'V Booster Set 02: Strongest! Team AL4'},
   {'ns': 0, 'exists': '', '*': 'Accel'},
   {'ns': 0, 'exists': '', '*': 'Aichi Sendou (V Series)'},
   {'ns': 0, 'exists': '', '*': 'Akira Ito'},
   {'ns': 0, 'exists': '', '*': 'Asura Kaiser (V Series)'},
   {'ns': 0, 'exists': '', '*': 'Battle Maiden, Sarasa'},
   {'ns': 0, 'exists': '', '*': 'Battledore Fighter'},
   {'ns': 0, 'exists': '', '*': 'Battleraizer (V Series)'},
   {'ns': 0, 'exists': '', '*': 'Bellicosity Dragon (V Series)'},
   {'ns': 0, 'exists': '', '*': 'Berserk Dragon (V Series)'},
   {'ns': 0, 'exists': '', '*': 'Blaster Blade (V Series)'},
   {'ns': 0, 'exists': '', '*': 'Boomerang Thrower (V Series)'},
  

In [ ]:
from bs4 import BeautifulSoup

In [ ]:
soup = BeautifulSoup(api_answer["parse"]["text"]["*"], "html.parser")

In [7]:
wikitex = pipeline.scrapper.obtain_wikitex(api_answer)
data = pipeline.scrapper.make_cardlist_from_str(wikitex=wikitex)
infobox = pipeline.parser.infobox(wikitex)

In [32]:
data[0].params[1].value

'King of Knights, Alfred'

In [10]:
wiki = pipeline.scrapper.obtain_links(api_answer)

In [12]:
pipeline.parser.clean_trash_from_set(state_machine.data["page"], wiki, 4, reverse=True)

In [54]:
data[0].params[1].value

'King of Knights, Alfred'

In [61]:
state_machine.data

{'answer': 'boosters',
 'category': 'boosters',
 'subcategory': 'Booster Sets',
 'param': {'action': 'parse',
  'page': 'List of Cardfight!! Vanguard Booster Sets',
  'format': 'json'},
 'page': 'List of Cardfight!! Vanguard Booster Sets',
 'response': {'parse': {'title': 'List of Cardfight!! Vanguard Booster Sets',
   'pageid': 2586,
   'revid': 3037270,
   'text': {'*': '<div class="mw-content-ltr mw-parser-output" lang="en" dir="ltr"><table cellspacing="0" class="navbox" style="border-spacing:0;;"><tbody><tr><td style="padding:2px;"><table cellspacing="0" class="nowraplinks collapsible uncollapsed navbox-inner" style="border-spacing:0;background:transparent;color:inherit;;"><tbody><tr><th scope="col" style=";" class="navbox-title" colspan="2"><span class="noprint plainlinks navbar" style=""><span style="white-space:nowrap;word-spacing:-.12em;"><a href="/wiki/Template:CardSets" title="Template:CardSets"><span style=";;background:none transparent;border:none;" title="View this templat

In [1]:
type(wiki[0])

NameError: name 'wiki' is not defined

In [58]:
link_in_wiki = []
i = 0
ll = len(data)
for i in range(len(data)):
	for z in wiki:
		if (str(data[i].params[1].value) in z):
			print(z)
			break
	link_in_wiki.append(z)
# for i in wiki:
# 	if (str(data[0].params[1].value) in i):
# 		print("XD")
# 		break

King of Knights, Alfred (V Series)
Imperial Daughter (V Series)
Dragonic Waterfall (V Series)
Perfect Raizer (V Series)
Soul Saver Dragon (V Series)
High Dog Breeder, Akane (V Series)
CEO Amaterasu (V Series)
Silent Tom (V Series)
Circle Magus (V Series)
Berserk Dragon (V Series)
Flame of Hope, Aermo (V Series)
Asura Kaiser (V Series)
Conjurer of Mithril (V Series)
Little Sage, Marron (V Series)
Flash Shield, Iseult (V Series)
Victorious Deer
Promise Daughter (V Series)
Weather Forecaster, Miss Mist (V Series)
Cruel Dragon (V Series)
Dragonic Gaias (V Series)
Wyvern Guard, Barri (V Series)
Hi-powered Raizer Custom (V Series)
Hi-powered Raizer Custom (V Series)
Twin Blader (V Series)
Funelgal
Knight of Rose, Morgana (V Series)
Pongal (V Series)
Yellow Witch, MeMe
Goddess of Insight, Sotoorihime
Oracle Guardian, Gemini (V Series)
Farfalle Magus
Ruote Magus
Vortex Dragon (V Series)
Prowling Dragon, Striken (V Series)
Bellicosity Dragon (V Series)
Lizard Soldier, Raopia (V Series)
Battledo

In [8]:
param = {
    "action": "parse",
    "page": "V Booster Set 01: Unite! Team Q4",
    "prop": "links",
    "format": "json"
}

In [9]:
api_answer = await pipeline.scrapper.api.get(params=param, headers=header)

In [ ]:
api_answer

In [ ]:
print(pipeline.storage.prepare_data([data[0]], 6, infobox=infobox))

In [ ]:
df = pd.DataFrame(data, columns=[
	"CardNo", "Name", "Grade", "Faction",
	"FactionType", "Type", "Rarity", "Release"
])

In [ ]:
df

In [ ]:
await main_scrap_routine(pipeline.parser, pipeline.storage, pipeline.scrapper, pipeline.classifier)

In [ ]:
api_answer["query"]["pages"]["525299"]["revisions"][0]["slots"]["main"]["*"]

In [ ]:
api_result = await pipeline.scrapper.api.get(
	params=dz_consults[0],
	headers=header
)

In [ ]:
api_result

In [ ]:
api_answer = await pipeline.scrapper.api.get(params=dz_consults[0], headers=header)
wikitex = pipeline.scrapper.obtain_wikitex(api_answer)
data = pipeline.scrapper.make_cardlist_from_str(wikitex=wikitex)
infobox = pipeline.parser.infobox(wikitex)
rows = pipeline.storage.prepare_data(data, 5, is_d=True, infobox=infobox)
df = pd.DataFrame(rows, columns=[
	"CardNo", "Name", "Grade", "Faction",
	"FactionType", "Type", "Rarity", "Release"
])
df.to_parquet("data/database/wrong.parquet")

In [ ]:
df = pd.read_parquet("data/database/boosters/booster sets/v/set_-1_8.parquet")

In [ ]:
df